# Load Packages

In [38]:
import os
import pandas as pd
from transformers import pipeline

In [39]:
# Custom functions
from custom_functions import (
    get_device,
    get_pipeline_device_id,
    analyze_sentiment,
)

# Parameters

In [40]:
# Output directory — all plots and tables land here
output_dir = os.path.join("..", "outputs")
os.makedirs(output_dir, exist_ok=True)

# Data directory — all data files land here
data_raw_dir = os.path.join("..", "data", "raw")
data_processed_dir = os.path.join("..", "data", "processed")

# Load Processed Data

In [41]:
# Load dataset
toaster_dedup_df = pd.read_csv(os.path.join(data_processed_dir, "toaster_dedup.csv"))

# Data preview
toaster_dedup_df.head()

,ASIN,P_TITLE,OP,DP,SP,FS,PRA_4.5,P_RTG,RTG_P_NO,SELLER_LINK,...,RV_DT,VP,HLP_VT,IMG_PRST,TTL_RV,RVS_L,RV_TRANS,SUBJ,SRVS,CP_RVS
0,B01KZ729F6,"Hamilton Beach 2 Slice Extra Wide Slot Toaster with Bagel & Defrost Settings, Shade Selector, Toast Boost, Auto Shut...",NaN,0.220000,24\n.\n99,1,0,4.4,12579,https://www.amazon.com/stores/HamiltonBeach/page/61F77C7C-D7C2-4996-90DD-A99598994A35?ref_=ast_bln,...,2019-01-06,1,.,0,1669,31,Love making 4 slices at a time.,0.6,positive,0.6369
1,B0744M3SB4,Nostalgia TCS2 Grilled Cheese Toaster with Easy-Clean Toaster Baskets and Adjustable Toasting Dial,209.988477,0.786655,44.8,1,0,4.1,4156,https://www.amazon.com/stores/Nostalgia/page/BEF8C71C-F2C2-4777-8128-A4BCA210DCF5?ref_=ast_bln,...,2018-12-09,1,.,0,863,96,Great item to use if you love grilled cheese but you have to use thin sliced bread only downfall,0.675,positive,0.8519
2,B0BT5WXBR2,"Elite Gourmet ECT118B Cool Touch Single Slice Toaster, 6 Toasting Levels & Wide Slot for Bagels, Waffles, Specialty ...",14.990000,0.000000,14.99,1,0,.,.,https://www.amazon.com/stores/EliteGourmet/page/7C41F0D0-1198-4E29-9F0F-7F910F09B7C4?ref_=ast_bln,...,2021-03-28,1,.,0,4078,79,I had high hopes for this toaster - but it takes two cycles to brown properly.,0.32,positive,0.4404
3,B0B9MX21NV,"evoloop Toaster 2 Slice, Stainless Steel Bread Toasters with Warming Rack, 6 Bread Shade Settings,1.5"" Extra Wide Sl...",279.850020,0.874969,34.99,1,0,4.4,31,https://www.amazon.com/stores/evoloop/page/08727391-970A-46B7-8A8C-DCE335FCE37D?ref_=ast_bln,...,2022-11-17,0,1,1,61,3565,BE AWARE: Read all the instructions included as this warning exists there: the first time (and maybe the 2nd & 3rd t...,0.509831254980509,positive,0.9914
4,B00ZGCKSG8,"DASH Clear View Toaster: Extra Wide Slot Toaster with See Through Window - Defrost, Reheat + Auto Shut Off Feature f...",NaN,0.170000,41\n.\n48,1,0,4.4,11283,https://www.amazon.com/stores/DASH/page/F42BA37C-41D6-4B53-B178-A95B4E659D0B?ref_=ast_bln,...,2020-10-04,1,35,0,3224,.,.,.,.,.


In [42]:
toaster_dedup_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 62014 entries, 0 to 62013
Data columns (total 30 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ASIN         62014 non-null  str    
 1   P_TITLE      62014 non-null  str    
 2   OP           45643 non-null  float64
 3   DP           62014 non-null  float64
 4   SP           62014 non-null  str    
 5   FS           62014 non-null  str    
 6   PRA_4.5      62014 non-null  int64  
 7   P_RTG        62014 non-null  str    
 8   RTG_P_NO     62014 non-null  str    
 9   SELLER_LINK  62014 non-null  str    
 10  IMAGE_URL    62014 non-null  str    
 11  P_URL        62014 non-null  str    
 12  RV_URL       62014 non-null  str    
 13  PRFL_IMG     62014 non-null  str    
 14  PRFL_URL     62014 non-null  str    
 15  RV_TTL       62011 non-null  str    
 16  RVS          62008 non-null  str    
 17  RVR          62011 non-null  str    
 18  RSR          62014 non-null  int64  
 19  RVR_CONT     62

# Sentiment Analysis

In [43]:
# take a sample to speed up testing — remove or increase for full analysis
# toaster_dedup_df = toaster_dedup_df.sample(50, random_state=42).copy()

## Device Detection
This will help detect if there exist a GPU on the device.

In [44]:
device = get_device()
device_id = get_pipeline_device_id(device)

Using CUDA GPU: NVIDIA GeForce GTX 1050 Ti with Max-Q Design


## Load Sentiment Pipeline

In [45]:
# Label Mapping
LABEL_MAP = {
    # Standard text labels
    "positive": "positive",
    "negative": "negative",
    "neutral":  "neutral",
    # Numeric labels (verify order on the model's HuggingFace card!)
    "label_0":  "negative",
    "label_1":  "neutral",
    "label_2":  "positive",
}

In [46]:
# Different models to experiment with — each has its own strengths and weaknesses

# model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
# model_name = "distilbert-base-uncased-finetuned-sst-2-english"
# model_name = "cardiffnlp/twitter-roberta-base-sentiment"
# model_name = "microsoft/deberta-v3-base"
# model_name = "siebert/sentiment-roberta-large-english"

### Twitter ROBERTa Base (CardiffNLP model)

In [47]:

model_twrb = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_pipeline_twrb = pipeline(
    "sentiment-analysis",
    model=model_twrb,
    tokenizer=model_twrb,
    device=device_id,
    truncation=True,        # Handles long reviews (truncates to 512 tokens)
    max_length=512,
    batch_size=32,          # Process in batches for speed
)
# Analyze sentiment and add results to the DataFrame
toaster_dedup_df = analyze_sentiment(
    toaster_dedup_df,
    sentiment_pipeline=sentiment_pipeline_twrb,
    text_col="RV_TRANS",
    label_map=LABEL_MAP,
)

# Rename BERT output columns for clarity
toaster_dedup_df = toaster_dedup_df.rename(
    columns={
        "SENTIMENT": "TWRB_SENT",
        "SENTIMENT_SCORE": "TWRB_SCORE",
    }
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Analyzing 61395 texts (skipping 619 empty/invalid)...

Short texts (direct batch): 61305
Long texts (chunking + voting): 90



Long texts: 100%|██████████| 90/90 [00:23<00:00,  3.86it/s]


Done. 90 text(s) used chunking + voting.


In [48]:
# Check distribution of sentiment labels and review the long reviews that were chunked
print("\nSentiment Distribution:")
display(toaster_dedup_df["TWRB_SENT"].value_counts())

print("\nLong reviews that used chunking:")
display(toaster_dedup_df[toaster_dedup_df["CHUNKED"]][["RV_TRANS", "TWRB_SENT", "TWRB_SCORE"]])


Sentiment Distribution:


TWRB_SENT
positive    37406
negative    17124
neutral      6865
Name: count, dtype: int64


Long reviews that used chunking:


,RV_TRANS,TWRB_SENT,TWRB_SCORE
3,BE AWARE: Read all the instructions included as this warning exists there: the first time (and maybe the 2nd & 3rd t...,neutral,0.6821
196,[UPDATED at bottom of review]\n-------------------------------------\nWe are reserving final judgment on this until ...,neutral,0.8554
741,"I just looked at the date we bought this, it was about 15 months ago. Within the first few weeks, we realize the so...",negative,0.7893
938,"Over the years, before today, we had purchased three of the Breville 830XL toasters and our daughter has (on our rec...",neutral,0.6841
1159,"Like so many others, I've recently started baking my own sourdough bread. I like my bread toasted, and didn't know t...",positive,0.8307
...,...,...,...
56008,"I am a fussy toast person, I like it just so.\nI had a nice inexpensive toaster for years and then it started to bre...",positive,0.5268
58835,"I own the Black & Decker TO1303SBD toaster oven, which is a slightly older model. I bought it in April 2020, and so...",positive,0.697
59673,"I'd give it 5 stars but for a toaster this expensive, you'd think the crumb tray would be well built and sized prope...",negative,0.6492
60085,"I wouldn't say that I'm a toast connoisseur, but I appreciate nice toast when I have it. Over my life I've probably ...",positive,0.8089


### ROBERTa Large English (Siebert model)

In [49]:
model_srb = "siebert/sentiment-roberta-large-english"

sentiment_pipeline_srb = pipeline(
    "sentiment-analysis",
    model=model_srb,
    tokenizer=model_srb,
    device=device_id,
    truncation=True,        # Handles long reviews (truncates to 512 tokens)
    max_length=512,
    batch_size=32,          # Process in batches for speed
)
# Analyze sentiment and add results to the DataFrame
toaster_dedup_df = analyze_sentiment(
    toaster_dedup_df,
    sentiment_pipeline=sentiment_pipeline_srb,
    text_col="RV_TRANS",
    label_map=LABEL_MAP,
)

# Rename BERT output columns for clarity
toaster_dedup_df = toaster_dedup_df.rename(
    columns={
        "SENTIMENT": "SRB_SENT",
        "SENTIMENT_SCORE": "SRB_SCORE",
    }
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (845 > 512). Running this sequence through the model will result in indexing errors


Analyzing 61395 texts (skipping 619 empty/invalid)...

Short texts (direct batch): 61305
Long texts (chunking + voting): 90



Long texts: 100%|██████████| 90/90 [01:08<00:00,  1.31it/s]


Done. 90 text(s) used chunking + voting.


In [51]:
# Check distribution of sentiment labels and review the long reviews that were chunked
print("\nSentiment Distribution:")
display(toaster_dedup_df["SRB_SENT"].value_counts())

print("\nLong reviews that used chunking:")
display(toaster_dedup_df[toaster_dedup_df["CHUNKED"]][["RV_TRANS", "SRB_SENT", "SRB_SCORE"]])


Sentiment Distribution:


SRB_SENT
positive    40456
negative    20939
Name: count, dtype: int64


Long reviews that used chunking:


,RV_TRANS,SRB_SENT,SRB_SCORE
3,BE AWARE: Read all the instructions included as this warning exists there: the first time (and maybe the 2nd & 3rd t...,positive,0.9978
196,[UPDATED at bottom of review]\n-------------------------------------\nWe are reserving final judgment on this until ...,negative,0.9988
741,"I just looked at the date we bought this, it was about 15 months ago. Within the first few weeks, we realize the so...",negative,0.9995
938,"Over the years, before today, we had purchased three of the Breville 830XL toasters and our daughter has (on our rec...",positive,0.9986
1159,"Like so many others, I've recently started baking my own sourdough bread. I like my bread toasted, and didn't know t...",positive,0.9987
...,...,...,...
56008,"I am a fussy toast person, I like it just so.\nI had a nice inexpensive toaster for years and then it started to bre...",positive,0.9988
58835,"I own the Black & Decker TO1303SBD toaster oven, which is a slightly older model. I bought it in April 2020, and so...",positive,0.9975
59673,"I'd give it 5 stars but for a toaster this expensive, you'd think the crumb tray would be well built and sized prope...",negative,0.999
60085,"I wouldn't say that I'm a toast connoisseur, but I appreciate nice toast when I have it. Over my life I've probably ...",positive,0.9988


## Results

In [52]:
# Review the results
cols = ["RV_TRANS", "SRVS", "CP_RVS", "TWRB_SENT", "TWRB_SCORE", "SRB_SENT", "SRB_SCORE"]
pd.set_option("display.max_colwidth", 120)
display(toaster_dedup_df[cols].head())

,RV_TRANS,SRVS,CP_RVS,TWRB_SENT,TWRB_SCORE,SRB_SENT,SRB_SCORE
0,Love making 4 slices at a time.,positive,0.6369,positive,0.967,positive,0.9988
1,Great item to use if you love grilled cheese but you have to use thin sliced bread only downfall,positive,0.8519,positive,0.8974,positive,0.9982
2,I had high hopes for this toaster - but it takes two cycles to brown properly.,positive,0.4404,negative,0.7371,negative,0.9995
3,BE AWARE: Read all the instructions included as this warning exists there: the first time (and maybe the 2nd & 3rd t...,positive,0.9914,neutral,0.6821,positive,0.9978
4,.,.,.,<NA>,<NA>,<NA>,<NA>


## Save Results to File

In [70]:
toaster_dedup_df.to_csv(os.path.join(data_processed_dir, "toaster_sentiment.csv"), index=False)

# Compare Results

In [54]:
# Extract relevant columns for comparison
compare_df = toaster_dedup_df[["SRVS", "TWRB_SENT", "SRB_SENT", "RV_TRANS"]].copy()

# Normalize text labels
compare_df["SRVS_clean"] = (
    compare_df["SRVS"]
    .astype("string")
    .str.strip()
    .str.lower()
)

compare_df["TWRB_SENT_clean"] = (
    compare_df["TWRB_SENT"]
    .astype("string")
    .str.strip()
    .str.lower()
)

compare_df["SRB_SENT_clean"] = (
    compare_df["SRB_SENT"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [60]:
# Keep only rows where both columns are available
compare_valid = compare_df.dropna(subset=["SRVS_clean", "TWRB_SENT_clean", "SRB_SENT_clean"]).copy()

# Check match between SRVS and TWRB_SENT
compare_valid["MATCH_SRVS_TWRB"] = (
    compare_valid["SRVS_clean"] == compare_valid["TWRB_SENT_clean"]
)

# Check match between SRVS and SRB_SENT
compare_valid["MATCH_SRVS_SRB"] = (
    compare_valid["SRVS_clean"] == compare_valid["SRB_SENT_clean"]
)

# Check match between TWRB_SENT and SRB_SENT
compare_valid["MATCH_TWRB_SRB"] = (
    compare_valid["TWRB_SENT_clean"] == compare_valid["SRB_SENT_clean"]
)


# Check match betwween all three
compare_valid["MATCH_ALL"] = (
    (compare_valid["SRVS_clean"] == compare_valid["TWRB_SENT_clean"]) &
    (compare_valid["SRVS_clean"] == compare_valid["SRB_SENT_clean"]) 
)

print(f"Number of valid rows: {len(compare_valid)}")

Number of valid rows: 61395


## VADER vs Twitter ROBERTa Base

In [56]:
# Agreement rate between SRVS and TWRB_SENT
agreement_rate_SRVS_TWRB = compare_valid["MATCH_SRVS_TWRB"].mean()

print(f"Agreement rate between SRVS and TWRB_SENT: {agreement_rate_SRVS_TWRB:.2%}")

# Cross-tab comparison
pd.crosstab(
    compare_valid["SRVS_clean"],
    compare_valid["TWRB_SENT_clean"],
    margins=True
)

Agreement rate between SRVS and TWRB_SENT: 72.77%


TWRB_SENT_clean,negative,neutral,positive,All
SRVS_clean,,,,
negative,6561,835,762,8158
neutral,2541,2847,1377,6765
positive,8022,3183,35267,46472
All,17124,6865,37406,61395


## VADER vs ROBERTa Large English

In [58]:
# Agreement rate between SRVS and SRB_SENT
agreement_rate_SRVS_SRB = compare_valid["MATCH_SRVS_SRB"].mean()

print(f"Agreement rate between SRVS and SRB_SENT: {agreement_rate_SRVS_SRB:.2%}")

# Cross-tab comparison
pd.crosstab(
    compare_valid["SRVS_clean"],
    compare_valid["SRB_SENT_clean"],
    margins=True
)

Agreement rate between SRVS and SRB_SENT: 70.62%


SRB_SENT_clean,negative,positive,All
SRVS_clean,,,
negative,6932,1226,8158
neutral,3960,2805,6765
positive,10047,36425,46472
All,20939,40456,61395


## Twitter ROBERTa Base vs ROBERTa Large English

In [59]:
# Agreement rate between TWRB_SENT and SRB_SENT
agreement_rate_TWRB_SRB = compare_valid["MATCH_TWRB_SRB"].mean()

print(f"Agreement rate between TWRB_SENT and SRB_SENT: {agreement_rate_TWRB_SRB:.2%}")

# Cross-tab comparison
pd.crosstab(
    compare_valid["TWRB_SENT_clean"],
    compare_valid["SRB_SENT_clean"],
    margins=True
)

Agreement rate between TWRB_SENT and SRB_SENT: 84.54%


SRB_SENT_clean,negative,positive,All
TWRB_SENT_clean,,,
negative,16039,1085,17124
neutral,3357,3508,6865
positive,1543,35863,37406
All,20939,40456,61395


## All Three

In [69]:
# Agreement rate where all three match
agreement_rate_all = compare_valid["MATCH_ALL"].mean()
print(f"Agreement rate where all three match: {agreement_rate_all:.2%}")

# Cross-tab comparison
pd.crosstab(
    [compare_valid["SRB_SENT_clean"],
    compare_valid["TWRB_SENT_clean"]],
     compare_valid["SRVS_clean"],
    margins=True
)

Agreement rate where all three match: 65.49%


SRVS_clean                      negative  neutral  positive    All
SRB_SENT_clean TWRB_SENT_clean                                    
negative       negative             6280     2477      7282  16039
               neutral               532     1399      1426   3357
               positive              120       84      1339   1543
positive       negative              281       64       740   1085
               neutral               303     1448      1757   3508
               positive              642     1293     33928  35863
All                                 8158     6765     46472  61395